# Этап 1 — Разведочный анализ (EDA)

**Датасет:** Online Retail II (UCI) — транзакции британского интернет-магазина
за 2009–2010 гг., ~525 тыс. строк. Один ряд = одна позиция в чеке
(товар, количество, цена, дата, клиент, страна).

**Цель ноутбука:** изучить данные, найти проблемные строки и принять обоснованные
решения по очистке, чтобы на этапе 2 корректно посчитать RFM-метрики
(Recency / Frequency / Monetary) на каждого клиента.

**План:** обзор структуры → поиск проблем → анализ их природы и пересечений
→ решения по очистке.

In [74]:
# импорт нужных библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [75]:
# загрузка базы и проверка ее корректности
df = pd.read_excel("../data/raw/online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [76]:
# описание таблицы
# в двух столбцах есть отсутствующие данные
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [77]:
# количество пустых значений
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [78]:
# начинаю смотреть таблицы с проблемными строками, в том числе с отсутствующими данными

# просмотр таблицы с отсутствующими данными в столбце 'Customer ID'
# смущает отрицательный 'Quantity' и 'Price'= 0
print('=== Пропуски в Customer ID ===')
display(df[df['Customer ID'].isna()].sample(10, random_state=50))

# просмотр таблицы с отрицательными данными в колонке 'Quantity'
print('=== Отрицательные Quantity ===')
display(df[df['Quantity'] < 0].sample(10, random_state=50))

# просмотр таблицы со столбцом 'Price'= 0
print('=== Price равный 0 ===')
display(df[df['Price'] == 0].sample(10, random_state=50))

# просмотр таблицы с отсутствующими данными в столбце 'Description'
print('=== Пропуски в Description ===')
display(df[df['Description'].isna()].sample(10, random_state=60))

=== Пропуски в Customer ID ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
81143,496855,DCGS0056,NaN,-4,2010-02-04 11:46:00,0.00,NaN,United Kingdom
36420,492414,21986,PACK OF 12 PINK SPOT TISSUES,1,2009-12-16 17:01:00,0.83,NaN,United Kingdom
50370,494015,22347,PIZZA SLICE DISH,1,2010-01-11 09:34:00,2.57,NaN,United Kingdom
270617,515609,20914,SET/5 RED SPOTTY LID GLASS BOWLS,1,2010-07-13 15:44:00,5.91,NaN,United Kingdom
55341,494386,22355,"CHARLOTTE BAG , SUKI DESIGN",1,2010-01-14 09:41:00,1.66,NaN,United Kingdom
221373,510983,21676,FLOWERS STICKERS,1,2010-06-04 12:23:00,1.66,NaN,United Kingdom
42577,493066,22072,TEA CUP AND SAUCER RETRO SPOT,1,2009-12-21 17:14:00,3.75,NaN,United Kingdom
466029,533348,22190,LOCAL CAFE MUG,1,2010-11-17 09:23:00,2.51,NaN,United Kingdom
55225,494386,85061W,WHITE JEWELLED HEART DECORATION,3,2010-01-14 09:41:00,1.66,NaN,United Kingdom
95979,498441,84598,BOYS ALPHABET IRON ON PATCHES,9,2010-02-19 09:37:00,0.43,NaN,United Kingdom


=== Отрицательные Quantity ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
43745,C493190,22162,HEART GARLAND RUSTIC PADDED,-1,2009-12-22 13:08:00,2.95,15581.0,United Kingdom
101711,499048,37490D,NaN,-40,2010-02-24 13:38:00,0.00,NaN,United Kingdom
473125,C533970,22719,GUMBALL MONOCHROME COAT RACK,-7,2010-11-19 13:17:00,1.06,15311.0,United Kingdom
263325,C514832,22167,"WALL MIRROR , DIAMANTE OVAL SHAPE",-1,2010-07-06 14:00:00,9.95,17867.0,United Kingdom
215288,C510234,22634,CHILDS BREAKFAST SET SPACEBOY,-1,2010-05-28 09:42:00,9.95,14291.0,United Kingdom
111137,C500016,84750A,PINK SMALL GLASS CAKE STAND,-1,2010-03-04 10:34:00,1.95,12391.0,Cyprus
414639,C528984,71477,COLOUR GLASS. STAR T-LIGHT HOLDER,-1,2010-10-26 11:04:00,3.25,18247.0,United Kingdom
101716,499054,85085A,NaN,-30,2010-02-24 13:44:00,0.00,NaN,United Kingdom
220916,C510816,21621,VINTAGE UNION JACK BUNTING,-2,2010-06-03 19:44:00,8.50,16592.0,United Kingdom
231557,C511807,79323P,PINK CHERRY LIGHTS,-36,2010-06-10 15:29:00,5.45,17850.0,United Kingdom


=== Price равный 0 ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
431587,530625,72715,NaN,-2,2010-11-03 16:01:00,0.0,NaN,United Kingdom
291697,517814,21116,OWL DOORSTOP,3,2010-08-02 11:44:00,0.0,NaN,United Kingdom
80578,496777,85056,NaN,-2,2010-02-03 15:56:00,0.0,NaN,United Kingdom
47823,493845,84718,NaN,-95,2010-01-07 15:02:00,0.0,NaN,United Kingdom
193705,507832,35652,NaN,-1,2010-05-11 13:24:00,0.0,NaN,United Kingdom
341739,522753,79328,NaN,-50,2010-09-16 13:34:00,0.0,NaN,United Kingdom
303111,518937,82072,NaN,1,2010-08-12 14:46:00,0.0,NaN,United Kingdom
106661,499616,20731,NaN,46,2010-03-01 12:40:00,0.0,NaN,United Kingdom
117384,500543,37502,NaN,129,2010-03-08 15:31:00,0.0,NaN,United Kingdom
193985,507852,17084A,NaN,-32,2010-05-11 14:50:00,0.0,NaN,United Kingdom


=== Пропуски в Description ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
193639,507800,85050,NaN,48,2010-05-11 12:47:00,0.0,NaN,United Kingdom
45734,493527,35980C,NaN,200,2010-01-04 15:15:00,0.0,NaN,United Kingdom
94430,498267,21396,NaN,-342,2010-02-17 14:59:00,0.0,NaN,United Kingdom
53273,494255,84925D,NaN,214,2010-01-12 16:44:00,0.0,NaN,United Kingdom
408199,528463,21825,NaN,-40,2010-10-22 11:36:00,0.0,NaN,United Kingdom
339692,522505,20692,NaN,1,2010-09-15 11:25:00,0.0,NaN,United Kingdom
428290,530314,37330,NaN,-284,2010-11-02 13:28:00,0.0,NaN,United Kingdom
158105,504453,22528,NaN,4800,2010-04-13 15:34:00,0.0,NaN,United Kingdom
254918,513969,21767,NaN,1,2010-06-29 15:56:00,0.0,NaN,United Kingdom
70007,495597,18096C,NaN,2,2010-01-26 10:30:00,0.0,NaN,United Kingdom


In [79]:
no_id    = df['Customer ID'].isna()
neg_qty  = df['Quantity'] < 0
zero_p   = df['Price'] == 0
no_desc  = df['Description'].isna()
survives = df['Customer ID'].notna() & (df['Price'] > 0)   # строка переживёт очистку

problems = {
    'Пропуск Customer ID': no_id,
    'Quantity < 0':        neg_qty,
    'Price = 0':           zero_p,
    'Пропуск Description':  no_desc,
}

decision = {
    'Пропуск Customer ID': 'Удаляем — без клиента нет RFM',
    'Quantity < 0':        'Оставляем возвраты (снижают Monetary)',
    'Price = 0':           'Удаляем — нулевой вклад в Monetary',
    'Пропуск Description':  'Игнорируем — все и так удаляются',
}

# идём по проблемам по одной и собираем строки таблицы
rows = []
for name, mask in problems.items():
    rows.append({
        'Проблема':        name,
        'Строк':           mask.sum(),
        'Доля, %':         round(mask.mean() * 100, 2),
        'без Customer ID': (mask & no_id).sum(),
        'Price = 0':       (mask & zero_p).sum(),
        'нет Description':  (mask & no_desc).sum(),
        'доживёт до RFM':  (mask & survives).sum(),
        'Решение':         decision[name],
    })

summary = pd.DataFrame(rows).set_index('Проблема')
summary

,Строк,"Доля, %",без Customer ID,Price = 0,нет Description,доживёт до RFM,Решение
Проблема,,,,,,,
Пропуск Customer ID,107927,20.54,107927,3656,2928,0,Удаляем — без клиента нет RFM
Quantity < 0,12326,2.35,2487,2121,1827,9839,Оставляем возвраты (снижают Monetary)
Price = 0,3687,0.70,3656,3687,2928,0,Удаляем — нулевой вклад в Monetary
Пропуск Description,2928,0.56,2928,2928,2928,0,Игнорируем — все и так удаляются


In [80]:
# Проверяем, все ли отрицательные Quantity — это возвраты клиентов (Invoice на 'C').
# ВАЖНО: Invoice — смешанный тип (str у отмен, int у обычных). Без astype(str)
# метод .str даёт NaN на числах, и value_counts их прячет — картина искажается.
df[df['Quantity'] < 0]['Invoice'].astype(str).str.startswith('C').value_counts()

Invoice
True     10205
False     2121
Name: count, dtype: int64

## Природа проблем и решения по очистке

Разобрали каждый тип проблемных строк, их пересечения и то, доживает ли
он до расчёта RFM.

**Пропуски в Customer ID — 107 927 (20.5 %).**
Транзакция есть, но клиент не идентифицирован. Для RFM это критично:
метрики считаются на клиента, без ID покупку не к кому привязать.
→ **Удаляем.**

**Отрицательный Quantity — 12 326 (2.3 %).** Делим по наличию Customer ID:
- **9 839** — возвраты клиентов (есть Customer ID, Invoice на «C») — **оставляем**;
- **2 487** — строки без Customer ID: из них 2 121 складские корректировки
  (числовой Invoice, Price = 0) и 366 анонимных возвратов (Invoice на «C»).
  Все 2 487 удаляются вместе с пропусками Customer ID.

Возвраты клиентов оставляем: их отрицательный вклад в Monetary
(`Price × Quantity < 0`) корректно снижает реальную ценность клиента.
→ **Возвраты клиентов оставляем, строки без ID уходят сами.**

*(Ячейка выше делит те же 12 326 иначе — по типу Invoice: 10 205 с «C» и
2 121 числовых. Это деление по типу счёта, а не по наличию клиента,
поэтому числа не совпадают с 9 839 / 2 487.)*

**Нулевая цена — Price = 0 — 3 687 (0.7 %).**
Отдельный признак, не подмножество возвратов: бывает и при `Quantity > 0`
(бесплатные/промо-позиции, не проставленная цена), и при `Quantity < 0`.
Общее одно: вклад в Monetary = 0. Плюс 3 656 из 3 687 и так без клиента.
→ **Удаляем** (причина — нулевой денежный вклад, а не «возврат»).

**Пропуски в Description — 2 928 (0.6 %).**
Все 2 928 одновременно имеют Price = 0 и пустой Customer ID — то есть
пропуск описания встречается только у служебных строк, не у реальных
продаж. Отдельная обработка не нужна: все удаляются фильтрами выше
(доживает до RFM — ноль).
→ **Игнорируем** (Description в RFM не используется и уходит сам).

---

**Про пересечения.** Эти признаки не независимы, а сильно
**перекрываются**: одна служебная строка часто битая сразу по нескольким
причинам (нет Customer ID + Price = 0 + Quantity < 0 + нет Description).
Поэтому нельзя складывать счётчики проблем — суммарно удаляется меньше,
чем `107 927 + 12 326 + 3 687`.

**Итог очистки.** Применяя два фильтра (удалить пропуски Customer ID,
затем удалить Price = 0), из **525 461** строк получаем **417 503**:

- −107 927 — строки без Customer ID;
- −31 — оставшиеся с Price = 0 (всего таких 3 687, но 3 656 уже ушли
  вместе с пропусками Customer ID — проблемы пересекаются, поэтому второй
  фильтр убирает лишь 31 «уникальную» строку).

Осознанно сохранено **9 839** возвратов клиентов (Quantity < 0, но с
Customer ID и Price > 0 — колонка «доживёт до RFM» в таблице выше). Они
уменьшают Monetary, отражая реальную ценность клиента. Сама очистка
применяется на этапе 2 (`02_rfm.ipynb`).